# Superfermion vs Qiskit vs PennyLane

Cross-framework benchmark comparing accuracy and speed on identical quantum circuits.

**Circuit families:** Bell, GHZ, QFT, Random Hardware-Efficient  
**Metrics:** Statevector fidelity, total variation distance (TVD), wall-clock time  
**Convention:** All bitstrings big-endian (qubit 0 = MSB / leftmost)

In [ ]:
import time
import math
import tracemalloc
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

import superfermion as sf
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import pennylane as qml

print(f"Superfermion {sf.__version__}")
import qiskit; print(f"Qiskit {qiskit.__version__}")
print(f"PennyLane  {qml.__version__}")
print(f"NumPy      {np.__version__}")

## 1. Helpers

In [ ]:
def fidelity(sv1: np.ndarray, sv2: np.ndarray) -> float:
    """State fidelity |<psi1|psi2>|^2."""
    return float(np.abs(np.vdot(sv1, sv2)) ** 2)


def tvd(counts1: dict, counts2: dict) -> float:
    """Total variation distance between two count distributions."""
    keys = set(counts1) | set(counts2)
    t1 = sum(counts1.values())
    t2 = sum(counts2.values())
    if t1 == 0 or t2 == 0:
        return 1.0
    return 0.5 * sum(abs(counts1.get(k, 0) / t1 - counts2.get(k, 0) / t2) for k in keys)


def timed(fn, n_warmup=2, n_timed=10):
    """Time a callable with warmup. Returns (result, stats_dict_ms)."""
    for _ in range(n_warmup):
        fn()
    times = []
    for _ in range(n_timed):
        t0 = time.perf_counter()
        result = fn()
        times.append((time.perf_counter() - t0) * 1000)
    arr = np.array(times)
    stats = {
        "mean_ms": float(np.mean(arr)),
        "std_ms": float(np.std(arr)),
        "min_ms": float(np.min(arr)),
        "p50_ms": float(np.median(arr)),
    }
    return result, stats


def reverse_qubits_sv(sv: np.ndarray) -> np.ndarray:
    """Convert statevector from little-endian (Qiskit) to big-endian qubit order."""
    n = int(np.log2(len(sv)))
    return sv.reshape([2] * n).transpose(range(n - 1, -1, -1)).flatten()


def normalize_counts(counts: dict, n_qubits: int) -> dict:
    """Ensure all keys are zero-padded big-endian bitstrings."""
    out = {}
    for k, v in counts.items():
        key = str(k).replace(' ', '')
        key = key.zfill(n_qubits)
        out[key] = out.get(key, 0) + int(v)
    return out


print("Helpers loaded.")

## 2. Circuit Builders

Each builder returns circuits for all three frameworks built from the same gate sequence.

In [ ]:
# --- Bell ---
def bell_sf(n=2):
    return sf.Circuit(n).h(0).cnot(0, 1)

def bell_qiskit(n=2):
    qc = QuantumCircuit(n)
    qc.h(0)
    qc.cx(0, 1)
    return qc

def bell_pennylane(n=2):
    def circuit():
        qml.Hadamard(wires=0)
        qml.CNOT(wires=[0, 1])
    return circuit


# --- GHZ ---
def ghz_sf(n):
    c = sf.Circuit(n).h(0)
    for i in range(n - 1):
        c.cnot(i, i + 1)
    return c

def ghz_qiskit(n):
    qc = QuantumCircuit(n)
    qc.h(0)
    for i in range(n - 1):
        qc.cx(i, i + 1)
    return qc

def ghz_pennylane(n):
    def circuit():
        qml.Hadamard(wires=0)
        for i in range(n - 1):
            qml.CNOT(wires=[i, i + 1])
    return circuit


# --- QFT ---
def qft_sf(n):
    c = sf.Circuit(n)
    for i in range(n):
        c.h(i)
        for j in range(i + 1, n):
            c.cp(math.pi / (2 ** (j - i)), i, j)
    for i in range(n // 2):
        c.swap(i, n - 1 - i)
    return c

def qft_qiskit(n):
    qc = QuantumCircuit(n)
    for i in range(n):
        qc.h(i)
        for j in range(i + 1, n):
            qc.cp(math.pi / (2 ** (j - i)), i, j)
    for i in range(n // 2):
        qc.swap(i, n - 1 - i)
    return qc

def qft_pennylane(n):
    def circuit():
        for i in range(n):
            qml.Hadamard(wires=i)
            for j in range(i + 1, n):
                qml.ControlledPhaseShift(math.pi / (2 ** (j - i)), wires=[i, j])
        for i in range(n // 2):
            qml.SWAP(wires=[i, n - 1 - i])
    return circuit


# --- Random Hardware-Efficient ---
def _random_angles(n, depth, seed):
    rng = np.random.default_rng(seed)
    return rng.uniform(0, 2 * np.pi, size=(depth, n, 3))

def random_he_sf(n, depth=4, seed=42):
    angles = _random_angles(n, depth, seed)
    c = sf.Circuit(n)
    for d in range(depth):
        for q in range(n):
            c.rz(float(angles[d, q, 0]), q)
            c.rx(float(angles[d, q, 1]), q)
            c.rz(float(angles[d, q, 2]), q)
        for q in range(0, n - 1, 2):
            c.cz(q, q + 1)
        for q in range(1, n - 1, 2):
            c.cz(q, q + 1)
    return c

def random_he_qiskit(n, depth=4, seed=42):
    angles = _random_angles(n, depth, seed)
    qc = QuantumCircuit(n)
    for d in range(depth):
        for q in range(n):
            qc.rz(float(angles[d, q, 0]), q)
            qc.rx(float(angles[d, q, 1]), q)
            qc.rz(float(angles[d, q, 2]), q)
        for q in range(0, n - 1, 2):
            qc.cz(q, q + 1)
        for q in range(1, n - 1, 2):
            qc.cz(q, q + 1)
    return qc

def random_he_pennylane(n, depth=4, seed=42):
    angles = _random_angles(n, depth, seed)
    def circuit():
        for d in range(depth):
            for q in range(n):
                qml.RZ(float(angles[d, q, 0]), wires=q)
                qml.RX(float(angles[d, q, 1]), wires=q)
                qml.RZ(float(angles[d, q, 2]), wires=q)
            for q in range(0, n - 1, 2):
                qml.CZ(wires=[q, q + 1])
            for q in range(1, n - 1, 2):
                qml.CZ(wires=[q, q + 1])
    return circuit


CIRCUIT_FAMILIES = {
    "Bell":      (bell_sf, bell_qiskit, bell_pennylane, [2]),
    "GHZ":       (ghz_sf, ghz_qiskit, ghz_pennylane, [4, 8, 12, 16, 20]),
    "QFT":       (qft_sf, qft_qiskit, qft_pennylane, [4, 8, 12, 16, 20]),
    "Random-HE": (random_he_sf, random_he_qiskit, random_he_pennylane, [4, 8, 12, 16, 20]),
}

print(f"Defined {len(CIRCUIT_FAMILIES)} circuit families.")

## 3. Framework Runners

Unified interface: each runner returns `(statevector, counts)` given a circuit and parameters.

In [ ]:
AER_SIM = AerSimulator(method="statevector")


def run_sf(circuit, n_qubits, shots=4096):
    r = sf.run(circuit, device="statevector", shots=shots)
    sv = np.array(r.statevector, dtype=np.complex128) if r.statevector is not None else None
    counts = normalize_counts(r.counts, n_qubits)
    return sv, counts


def run_qiskit(circuit, n_qubits, shots=4096):
    # Statevector (Qiskit uses little-endian; reverse to big-endian for comparison)
    sv_circ = circuit.copy()
    sv_circ.save_statevector()
    sv_result = AER_SIM.run(sv_circ, shots=0).result()
    sv = reverse_qubits_sv(np.array(sv_result.get_statevector(), dtype=np.complex128))

    # Sampling
    meas_circ = circuit.copy()
    meas_circ.measure_all()
    counts_result = AER_SIM.run(meas_circ, shots=shots).result()
    counts = normalize_counts(counts_result.get_counts(), n_qubits)
    return sv, counts


def run_pennylane(circuit_fn, n_qubits, shots=4096):
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev)
    def sv_circuit():
        circuit_fn()
        return qml.state()

    sv = np.array(sv_circuit(), dtype=np.complex128)

    @qml.qnode(dev)
    def counts_circuit():
        circuit_fn()
        return qml.counts()

    raw_counts = qml.set_shots(counts_circuit, shots=shots)()
    counts = normalize_counts({str(k): int(v) for k, v in raw_counts.items()}, n_qubits)
    return sv, counts


print("Runners loaded.")

## 4. Accuracy Benchmark

Compare statevector fidelity and sampling TVD across all three frameworks.

In [ ]:
ACCURACY_QUBITS = {"Bell": [2], "GHZ": [4, 8, 12], "QFT": [4, 8, 12], "Random-HE": [4, 8, 12]}
SHOTS = 8192

accuracy_rows = []

for family, (sf_fn, qk_fn, pl_fn, _) in CIRCUIT_FAMILIES.items():
    for n in ACCURACY_QUBITS.get(family, []):
        print(f"  {family} n={n}...", end=" ", flush=True)

        sf_sv, sf_counts = run_sf(sf_fn(n), n, shots=SHOTS)
        qk_sv, qk_counts = run_qiskit(qk_fn(n), n, shots=SHOTS)
        pl_sv, pl_counts = run_pennylane(pl_fn(n), n, shots=SHOTS)

        # Fidelities (all pairs)
        f_sq = fidelity(sf_sv, qk_sv)
        f_sp = fidelity(sf_sv, pl_sv)
        f_qp = fidelity(qk_sv, pl_sv)

        # TVD on counts (all pairs)
        t_sq = tvd(sf_counts, qk_counts)
        t_sp = tvd(sf_counts, pl_counts)
        t_qp = tvd(qk_counts, pl_counts)

        row = {
            "family": family, "n_qubits": n,
            "fid_SF_QK": f_sq, "fid_SF_PL": f_sp, "fid_QK_PL": f_qp,
            "tvd_SF_QK": t_sq, "tvd_SF_PL": t_sp, "tvd_QK_PL": t_qp,
        }
        accuracy_rows.append(row)
        print(f"fidelity(SF,QK)={f_sq:.8f}  TVD(SF,QK)={t_sq:.4f}")

print(f"\n{len(accuracy_rows)} accuracy data points collected.")

In [ ]:
# Print accuracy table
header = f"{'Circuit':<14} {'n':>3} | {'Fid SF-QK':>10} {'Fid SF-PL':>10} {'Fid QK-PL':>10} | {'TVD SF-QK':>10} {'TVD SF-PL':>10} {'TVD QK-PL':>10}"
print(header)
print("-" * len(header))
for r in accuracy_rows:
    print(f"{r['family']:<14} {r['n_qubits']:>3} | "
          f"{r['fid_SF_QK']:>10.8f} {r['fid_SF_PL']:>10.8f} {r['fid_QK_PL']:>10.8f} | "
          f"{r['tvd_SF_QK']:>10.4f} {r['tvd_SF_PL']:>10.4f} {r['tvd_QK_PL']:>10.4f}")

## 5. Speed Benchmark

Measure execution time for statevector simulation across qubit counts.

In [ ]:
SPEED_FAMILIES = ["GHZ", "QFT", "Random-HE"]
SPEED_QUBITS = [4, 8, 12, 16, 20]

speed_results = defaultdict(lambda: defaultdict(dict))  # family -> framework -> n -> stats

for family in SPEED_FAMILIES:
    sf_fn, qk_fn, pl_fn, _ = CIRCUIT_FAMILIES[family]
    print(f"\n=== {family} ===")

    for n in SPEED_QUBITS:
        print(f"  n={n:>2}: ", end="", flush=True)

        # Superfermion
        sf_circ = sf_fn(n)
        _, sf_stats = timed(lambda c=sf_circ: sf.run(c, device="statevector", shots=0))
        speed_results[family]["Superfermion"][n] = sf_stats
        print(f"SF={sf_stats['mean_ms']:>8.2f}ms  ", end="", flush=True)

        # Qiskit
        qk_circ = qk_fn(n)
        qk_circ_sv = qk_circ.copy()
        qk_circ_sv.save_statevector()
        _, qk_stats = timed(lambda c=qk_circ_sv: AER_SIM.run(c, shots=0).result())
        speed_results[family]["Qiskit"][n] = qk_stats
        print(f"QK={qk_stats['mean_ms']:>8.2f}ms  ", end="", flush=True)

        # PennyLane
        pl_fn_n = pl_fn(n)
        dev = qml.device("default.qubit", wires=n)
        @qml.qnode(dev)
        def pl_circuit(fn=pl_fn_n):
            fn()
            return qml.state()
        _, pl_stats = timed(pl_circuit)
        speed_results[family]["PennyLane"][n] = pl_stats
        print(f"PL={pl_stats['mean_ms']:>8.2f}ms")

print("\nSpeed benchmark complete.")

In [ ]:
# Print speed table
print(f"{'Family':<12} {'n':>3} | {'SF mean':>10} {'SF p50':>10} | {'QK mean':>10} {'QK p50':>10} | {'PL mean':>10} {'PL p50':>10}")
print("-" * 90)
for family in SPEED_FAMILIES:
    for n in SPEED_QUBITS:
        s = speed_results[family]
        sf_s = s["Superfermion"][n]
        qk_s = s["Qiskit"][n]
        pl_s = s["PennyLane"][n]
        print(f"{family:<12} {n:>3} | "
              f"{sf_s['mean_ms']:>8.2f}ms {sf_s['p50_ms']:>8.2f}ms | "
              f"{qk_s['mean_ms']:>8.2f}ms {qk_s['p50_ms']:>8.2f}ms | "
              f"{pl_s['mean_ms']:>8.2f}ms {pl_s['p50_ms']:>8.2f}ms")

## 6. Scaling Plots

In [ ]:
fig, axes = plt.subplots(1, len(SPEED_FAMILIES), figsize=(6 * len(SPEED_FAMILIES), 5), sharey=False)
colors = {"Superfermion": "#7c3aed", "Qiskit": "#2563eb", "PennyLane": "#059669"}
markers = {"Superfermion": "o", "Qiskit": "s", "PennyLane": "^"}

for ax, family in zip(axes, SPEED_FAMILIES):
    for fw in ["Superfermion", "Qiskit", "PennyLane"]:
        data = speed_results[family][fw]
        ns = sorted(data.keys())
        means = [data[n]["mean_ms"] for n in ns]
        stds = [data[n]["std_ms"] for n in ns]
        ax.errorbar(ns, means, yerr=stds, label=fw,
                    color=colors[fw], marker=markers[fw],
                    linewidth=2, markersize=7, capsize=3)
    ax.set_title(family, fontsize=14, fontweight="bold")
    ax.set_xlabel("Qubits")
    ax.set_ylabel("Time (ms)")
    ax.set_yscale("log")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle("Statevector Simulation Time vs Qubit Count", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("speed_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to speed_comparison.png")

## 7. Memory Benchmark

In [ ]:
MEM_QUBITS = 20
MEM_FAMILY = "Random-HE"
sf_fn, qk_fn, pl_fn, _ = CIRCUIT_FAMILIES[MEM_FAMILY]

print(f"Memory benchmark: {MEM_FAMILY} n={MEM_QUBITS}\n")

# Superfermion
tracemalloc.start()
sf.run(sf_fn(MEM_QUBITS), device="statevector", shots=0)
sf_peak = tracemalloc.get_traced_memory()[1] / (1024 * 1024)
tracemalloc.stop()

# Qiskit
tracemalloc.start()
qk_c = qk_fn(MEM_QUBITS)
qk_c.save_statevector()
AER_SIM.run(qk_c, shots=0).result()
qk_peak = tracemalloc.get_traced_memory()[1] / (1024 * 1024)
tracemalloc.stop()

# PennyLane
tracemalloc.start()
dev = qml.device("default.qubit", wires=MEM_QUBITS)
@qml.qnode(dev)
def pl_mem_circuit():
    pl_fn(MEM_QUBITS)()
    return qml.state()
pl_mem_circuit()
pl_peak = tracemalloc.get_traced_memory()[1] / (1024 * 1024)
tracemalloc.stop()

print(f"  Superfermion: {sf_peak:>8.2f} MB peak")
print(f"  Qiskit:       {qk_peak:>8.2f} MB peak")
print(f"  PennyLane:    {pl_peak:>8.2f} MB peak")

## 8. Summary

In [ ]:
print("="*70)
print("ACCURACY SUMMARY")
print("="*70)
all_fid = [r["fid_SF_QK"] for r in accuracy_rows] + [r["fid_SF_PL"] for r in accuracy_rows]
all_tvd = [r["tvd_SF_QK"] for r in accuracy_rows] + [r["tvd_SF_PL"] for r in accuracy_rows]
print(f"  Min fidelity (SF vs others): {min(all_fid):.10f}")
print(f"  Max TVD (SF vs others):      {max(all_tvd):.6f}")
print(f"  Mean TVD (SF vs others):     {np.mean(all_tvd):.6f}")
if min(all_fid) > 0.99999999:
    print("  -> PASS: Statevectors match to machine precision.")
else:
    print(f"  -> NOTE: Some fidelity < 1.0 detected.")

print()
print("="*70)
print("SPEED SUMMARY (median ms, n=20)")
print("="*70)
for family in SPEED_FAMILIES:
    s = speed_results[family]
    if 20 in s["Superfermion"]:
        sf_t = s["Superfermion"][20]["p50_ms"]
        qk_t = s["Qiskit"][20]["p50_ms"]
        pl_t = s["PennyLane"][20]["p50_ms"]
        fastest = min(sf_t, qk_t, pl_t)
        print(f"  {family:<12}: SF={sf_t:>8.1f}ms  QK={qk_t:>8.1f}ms  PL={pl_t:>8.1f}ms  "
              f"(SF {'fastest' if sf_t == fastest else f'{sf_t/fastest:.1f}x'})")

print()
print("="*70)
print("MEMORY SUMMARY")
print("="*70)
print(f"  {MEM_FAMILY} n={MEM_QUBITS}: SF={sf_peak:.1f}MB  QK={qk_peak:.1f}MB  PL={pl_peak:.1f}MB")
print()
print("Benchmark complete.")

## 9. Rust Core Optimizations Benchmark

Benchmarks for features introduced in the Rust Core Performance Audit:
- **Adjoint Differentiation**: O(M·2^n) gradient computation vs O(M·N·2^n) parameter-shift
- **Rust-side MSB conversion**: Endianness handled in Rust before returning to Python
- **Rust-side sampling**: Bitstring sampling without returning full statevector to Python
- **Raw Rust Core**: Direct `_sf_core.QuantumDAG.simulate()` bypassing Python pipeline overhead

In [ ]:
import time, math
import numpy as np
import _sf_core
import superfermion as sf
from superfermion.observables.core import SparsePauliOp

def timed_quick(fn, n_warmup=2, n_timed=5):
    for _ in range(n_warmup):
        fn()
    times = []
    for _ in range(n_timed):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times))

# ── 9a. Adjoint Gradient vs Parameter-Shift ──────────────────────────────
print("=" * 60)
print("ADJOINT GRADIENT vs PARAMETER-SHIFT")
print("=" * 60)

from superfermion.qml.gradient import parameter_shift, adjoint

for n_qubits, depth in [(4, 3), (6, 3), (8, 2)]:
    n_params = n_qubits * depth
    params = {f"p{i}": float(np.random.uniform(0, 2*np.pi)) for i in range(n_params)}
    
    c = sf.Circuit(n_qubits)
    pidx = 0
    for d in range(depth):
        for q in range(n_qubits):
            c.ry(sf.param(f"p{pidx}"), q)
            pidx += 1
        for q in range(n_qubits - 1):
            c.cnot(q, q + 1)
    
    obs = SparsePauliOp.from_dict({"Z" * n_qubits: 1.0})
    names = list(params.keys())
    values = np.array([params[n] for n in names])
    
    ps_time = timed_quick(lambda: parameter_shift.parameter_shift_grad_vector(c, obs, names, values, backend="statevector"))
    adj_time = timed_quick(lambda: adjoint.adjoint_grad_vector(c, obs, names, values))
    
    print(f"  {n_qubits}q, {n_params} params: PS={ps_time:.1f}ms  Adjoint={adj_time:.1f}ms  Speedup={ps_time/adj_time:.1f}x")

# ── 9b. Raw Rust Core vs Full SF Pipeline vs Qiskit ──────────────────────
print()
print("=" * 60)
print("RAW RUST CORE vs FULL SF PIPELINE vs QISKIT AER")
print("=" * 60)

def ghz_raw(n):
    dag = _sf_core.QuantumDAG(n, 0)
    dag.add_gate("h", [0], [])
    for i in range(n-1):
        dag.add_gate("cx", [i, i+1], [])
    return dag

def ghz_sf(n):
    c = sf.Circuit(n).h(0)
    for i in range(n-1):
        c.cnot(i, i+1)
    return c

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
aer = AerSimulator(method="statevector")

def ghz_qk(n):
    qc = QuantumCircuit(n)
    qc.h(0)
    for i in range(n-1):
        qc.cx(i, i+1)
    qc.save_statevector()
    return qc

print(f"{'n':>3} | {'Rust Core':>10} | {'SF Pipeline':>12} | {'Qiskit Aer':>11} | {'Rust/Qiskit':>12}")
print("-" * 60)
for n in [4, 8, 12, 16, 20]:
    dag = ghz_raw(n)
    circ_sf = ghz_sf(n)
    circ_qk = ghz_qk(n)
    
    t_rust = timed_quick(lambda d=dag: d.simulate())
    t_sf = timed_quick(lambda c=circ_sf: sf.run(c, device="statevector", shots=0))
    t_qk = timed_quick(lambda c=circ_qk: aer.run(c, shots=0).result())
    
    ratio = t_qk / t_rust if t_rust > 0.001 else float('inf')
    print(f"{n:>3} | {t_rust:>8.3f}ms | {t_sf:>10.3f}ms | {t_qk:>9.3f}ms | {ratio:>10.1f}x")

# ── 9c. Rust-side Sampling ────────────────────────────────────────────────
print()
print("=" * 60)
print("RUST-SIDE SAMPLING vs PYTHON SAMPLING (10k shots)")
print("=" * 60)

shots = 10000
for n in [8, 12, 16, 20]:
    dag = ghz_raw(n)
    
    def python_sample(d=dag, n_q=n):
        sv = np.asarray(d.simulate())
        probs = np.abs(sv)**2
        indices = np.random.choice(len(probs), size=shots, p=probs)
        counts = {}
        for idx in indices:
            bs = format(idx, f"0{n_q}b")
            counts[bs] = counts.get(bs, 0) + 1
        return counts
    
    t_rust_s = timed_quick(lambda d=dag: d.simulate_and_sample(shots, 42))
    t_py_s = timed_quick(lambda: python_sample())
    print(f"  n={n:>2}: Rust={t_rust_s:.2f}ms  Python={t_py_s:.2f}ms  Speedup={t_py_s/t_rust_s:.1f}x")

print()
print("Rust core optimization benchmarks complete.")

## 10. Superfermion vs Qiskit vs PennyLane — Full Benchmark

Cross-framework comparison: statevector simulation time on identical random circuits.

**Frameworks:**
- **Superfermion CPU** — Rust + Rayon + AVX-2 multi-threaded
- **Superfermion GPU** — Rust + CUDA (cudarc, runtime kernel compilation)
- **Qiskit Aer** — C++ statevector simulator (production-grade)
- **PennyLane** — `default.qubit` (NumPy-based)

**Test machine:** Intel CPU (WSL2) + NVIDIA GeForce MX350 (2GB VRAM, sm_61, 384 CUDA cores)  
**Circuit:** Random hardware-efficient (H + CNOT chain + RZ(random) + CNOT chain + H)  
**Methodology:** 3 trials, median time, post-warmup (kernel compilation excluded)

In [ ]:
import time, json
import numpy as np
import superfermion as sf
from superfermion.devices.rust_device import RustDevice
from superfermion._sf_core import gpu_available, gpu_diagnose
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import pennylane as qml

print(f"GPU available: {gpu_available()}")
print(f"GPU status: {gpu_diagnose()}")
print()

aer_sim = AerSimulator(method='statevector')
TRIALS = 3

# Warmups
sf.run(sf.Circuit(4).h(0).cnot(0,1), device='gpu', shots=0)
RustDevice._result_cache.clear()
qc_w = QuantumCircuit(4); qc_w.h(0); qc_w.cx(0,1); qc_w.save_statevector()
aer_sim.run(qc_w).result()
dev_w = qml.device('default.qubit', wires=4)
@qml.qnode(dev_w)
def _warmup(): qml.Hadamard(0); return qml.state()
_warmup()

def make_circuit_sf(n):
    np.random.seed(42)
    c = sf.Circuit(n)
    for i in range(n): c.h(i)
    for i in range(n-1): c.cnot(i, i+1)
    for i in range(n): c.rz(np.random.uniform(0, 2*np.pi), i)
    for i in range(n-1): c.cnot(i, i+1)
    for i in range(n): c.h(i)
    return c

def make_circuit_qiskit(n):
    np.random.seed(42)
    qc = QuantumCircuit(n)
    for i in range(n): qc.h(i)
    for i in range(n-1): qc.cx(i, i+1)
    for i in range(n): qc.rz(np.random.uniform(0, 2*np.pi), i)
    for i in range(n-1): qc.cx(i, i+1)
    for i in range(n): qc.h(i)
    qc.save_statevector()
    return qc

def make_circuit_pl(n):
    np.random.seed(42)
    ops = []
    for i in range(n): ops.append(qml.Hadamard(wires=i))
    for i in range(n-1): ops.append(qml.CNOT(wires=[i, i+1]))
    for i in range(n): ops.append(qml.RZ(np.random.uniform(0, 2*np.pi), wires=i))
    for i in range(n-1): ops.append(qml.CNOT(wires=[i, i+1]))
    for i in range(n): ops.append(qml.Hadamard(wires=i))
    return ops

qubit_sizes = [12, 14, 16, 18, 20, 22, 24]
results = {}

print(f"{'N':>3} | {'SF CPU':>8} | {'SF GPU':>8} | {'Qiskit':>8} | {'PennyLane':>9} | Winner")
print("-" * 65)

for n in qubit_sizes:
    times = {'sf_cpu': [], 'sf_gpu': [], 'qiskit': [], 'pennylane': []}
    
    for _ in range(TRIALS):
        RustDevice._result_cache.clear()
        c = make_circuit_sf(n)
        t0 = time.perf_counter()
        sf.run(c, device='cpu', shots=0)
        times['sf_cpu'].append((time.perf_counter() - t0) * 1000)

        RustDevice._result_cache.clear()
        c = make_circuit_sf(n)
        t0 = time.perf_counter()
        sf.run(c, device='gpu', shots=0)
        times['sf_gpu'].append((time.perf_counter() - t0) * 1000)

        qc = make_circuit_qiskit(n)
        t0 = time.perf_counter()
        aer_sim.run(qc).result()
        times['qiskit'].append((time.perf_counter() - t0) * 1000)

        dev = qml.device('default.qubit', wires=n)
        ops = make_circuit_pl(n)
        @qml.qnode(dev)
        def circuit():
            for op in ops:
                qml.apply(op)
            return qml.state()
        t0 = time.perf_counter()
        circuit()
        times['pennylane'].append((time.perf_counter() - t0) * 1000)

    r = {k: float(np.median(v)) for k, v in times.items()}
    results[n] = r
    
    all_t = {'SF CPU': r['sf_cpu'], 'SF GPU': r['sf_gpu'], 'Qiskit': r['qiskit'], 'PL': r['pennylane']}
    winner = min(all_t, key=all_t.get)
    print(f"{n:3} | {r['sf_cpu']:6.1f}ms | {r['sf_gpu']:6.1f}ms | {r['qiskit']:6.1f}ms | {r['pennylane']:7.1f}ms | {winner}")

print()
for n in [18, 20, 22]:
    r = results[n]
    fastest_sf = min(r['sf_cpu'], r['sf_gpu'])
    label = 'GPU' if r['sf_gpu'] < r['sf_cpu'] else 'CPU'
    print(f"  {n}q: SF {label} is {r['qiskit']/fastest_sf:.1f}x vs Qiskit, "
          f"{r['pennylane']/fastest_sf:.1f}x vs PennyLane")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Execution time comparison
ax1.semilogy(qubit_sizes, cpu_times, 'b-o', label='CPU (Rust + Rayon + AVX-2)', linewidth=2)
ax1.semilogy(qubit_sizes, gpu_times, 'r-s', label='GPU (Rust + CUDA, MX350)', linewidth=2)
ax1.set_xlabel('Number of Qubits', fontsize=12)
ax1.set_ylabel('Execution Time (ms, log scale)', fontsize=12)
ax1.set_title('Superfermion: CPU vs GPU Simulation', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_xticks(qubit_sizes)

# Speedup
speedups = [c / g if g > 0 else 0 for c, g in zip(cpu_times, gpu_times)]
ax2.bar(qubit_sizes, speedups, color=['green' if s > 1 else 'gray' for s in speedups], width=1.5)
ax2.axhline(y=1, color='black', linestyle='--', alpha=0.5, label='Break-even')
ax2.set_xlabel('Number of Qubits', fontsize=12)
ax2.set_ylabel('GPU Speedup (x)', fontsize=12)
ax2.set_title('GPU Speedup over CPU (MX350, 2GB VRAM)', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_xticks(qubit_sizes)

plt.tight_layout()
plt.savefig('cpu_vs_gpu_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: cpu_vs_gpu_benchmark.png")

## New API Summary

The redesigned `sf.run()` API provides explicit, zero-magic control:

```python
import superfermion as sf

circuit = sf.Circuit(20).h(0).cnot(0, 1)

# device= controls WHERE (CPU or GPU)
sf.run(circuit, device="cpu")        # Rust multi-threaded (default)
sf.run(circuit, device="gpu")        # Rust CUDA GPU

# method= controls HOW (simulation algorithm)
sf.run(circuit, method="statevector")  # exact, 2^n memory (default)
sf.run(circuit, method="mps")          # tensor network, large circuits
sf.run(circuit, method="stabilizer")   # Clifford-only, exponentially fast

# QPU via explicit provider objects (no magic strings)
from superfermion.devices.ibm import IBMDevice
ibm = IBMDevice(token="...")
sf.run(circuit, device=ibm("ibm_fez"))

from superfermion.devices.ionq import IonQDevice
ionq = IonQDevice(api_key="...")
sf.run(circuit, device=ionq("ionq.aria-1"))
```